<a href="https://colab.research.google.com/github/noor00-ai/fly_rank_intern/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noor00-ai/fly_rank_intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

This baseline ranks content items that may need attention using two observed signals:

1. Staleness:
Older content with more days since last update receives a higher score.

2. CTR vs Position:
Content with good search position but low CTR receives a higher score.

The rule is decision-support only and identifies items for review.

Reason codes:
- STALE_CONTENT: Content has not been updated recently.
- LOW_CTR_POSITION: Content has low CTR compared with its ranking position.
- MIXED_SIGNAL: Both signals contribute.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


Signal 1: Content Staleness

I check whether days since last update creates meaningful groups.

Verdict:
CONFIRMED if older content appears frequently enough to be a useful review signal.

In [16]:
# Create staleness buckets

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1,90,180,365,10000],
    labels=[
        "Fresh",
        "Medium",
        "Old",
        "Very Old"
    ]
)

stale_table = (
    df["staleness_bucket"]
    .value_counts()
    .reset_index()
)

stale_table.columns = ["staleness_bucket","n"]

print(stale_table)

print("\nVerdict: CONFIRMED")
print(
    "Reason: days_since_last_update provides measurable groups "
    "for identifying potentially stale content."
)

  staleness_bucket      n
0            Fresh  20655
1           Medium   9171
2              Old    169
3         Very Old      5

Verdict: CONFIRMED
Reason: days_since_last_update provides measurable groups for identifying potentially stale content.


Signal 2: CTR vs Position

I compare ranking position with CTR.

Content in strong positions but with weak CTR may need review.

Verdict:
MIXED because CTR depends on user intent and traffic quality.

In [17]:
# Create CTR-position groups

def ctr_position_group(row):
    if row["avg_position"] <= 10 and row["ctr"] < 1:
        return "Good_Position_Low_CTR"
    elif row["avg_position"] <= 10 and row["ctr"] >= 1:
        return "Good_Position_Good_CTR"
    elif row["avg_position"] > 10 and row["ctr"] < 1:
        return "Low_Position_Low_CTR"
    else:
        return "Other"


df["ctr_position_bucket"] = df.apply(
    ctr_position_group,
    axis=1
)

ctr_table = (
    df["ctr_position_bucket"]
    .value_counts()
    .reset_index()
)

ctr_table.columns = [
    "ctr_position_bucket",
    "n"
]

print(ctr_table)

print("\nVerdict: MIXED")
print(
    "Reason: CTR is useful, but position and user intent "
    "can influence the result."
)

      ctr_position_bucket      n
0    Low_Position_Low_CTR  15265
1   Good_Position_Low_CTR  13026
2  Good_Position_Good_CTR   1162
3                   Other    547

Verdict: MIXED
Reason: CTR is useful, but position and user intent can influence the result.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Baseline Score

The score combines:

- Staleness score
- CTR-position problem score

Higher score means higher review priority.

The queue is written to:
work/outputs/baseline_action_score.csv

In [18]:


# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create baseline scores

df["stale_score"] = (
    df["days_since_last_update"]
    .fillna(0)
    / df["days_since_last_update"].max()
)


df["ctr_position_score"] = np.where(
    (df["avg_position"] <= 10) &
    (df["ctr"] < 1),
    1,
    0
)


# Final baseline score

df["baseline_score"] = (
    df["stale_score"] * 0.6 +
    df["ctr_position_score"] * 0.4
)


# Reason codes

def assign_reason(row):

    if row["ctr_position_score"] == 1 and row["stale_score"] > 0.5:
        return "MIXED_SIGNAL"

    elif row["ctr_position_score"] == 1:
        return "LOW_CTR_POSITION"

    elif row["stale_score"] > 0.5:
        return "STALE_CONTENT"

    else:
        return "NO_ACTION"


df["reason_code"] = df.apply(
    assign_reason,
    axis=1
)


# Actions

df["action"] = np.where(
    df["baseline_score"] >= 0.5,
    "REVIEW_CONTENT",
    "MONITOR"
)


# Rank queue

queue = (
    df.sort_values(
        "baseline_score",
        ascending=False
    )
)


queue["rank"] = range(
    1,
    len(queue)+1
)


# Save output

import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)


queue[
[
"rank",
"content_id",
"client_id",
"baseline_score",
"reason_code",
"action"
]
].to_csv(
"work/outputs/baseline_action_score.csv",
index=False
)


print("CSV created successfully")

queue.head(10)

CSV created successfully


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,staleness_bucket,ctr_position_bucket,stale_score,ctr_position_score,baseline_score,reason_code,action,rank
26242,content_55a5b1c46474,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,down,-88.5,Very Old,Good_Position_Low_CTR,1.000000,1,1.000000,MIXED_SIGNAL,REVIEW_CONTENT,1
24216,content_1b4ec72dafd4,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,down,-100.0,Very Old,Good_Position_Low_CTR,0.997319,1,0.998391,MIXED_SIGNAL,REVIEW_CONTENT,2
15608,content_06e19c6486b0,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1300.0,9163.0,...,flat,NaN,Old,Good_Position_Low_CTR,0.895442,1,0.937265,MIXED_SIGNAL,REVIEW_CONTENT,3
8631,content_e2b702f4f92b,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1246.0,8740.0,...,down,-72.7,Old,Good_Position_Low_CTR,0.895442,1,0.937265,MIXED_SIGNAL,REVIEW_CONTENT,4
21984,content_02b0d6e30129,client_19581e27de,110.0,0.40,MEDIUM,0.59,keyword article,transactional,NaN,NaN,...,down,-95.6,Old,Good_Position_Low_CTR,0.839142,1,0.903485,MIXED_SIGNAL,REVIEW_CONTENT,5
3723,content_f488400fca67,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1113.0,8048.0,...,down,-44.0,Old,Good_Position_Low_CTR,0.817694,1,0.890617,MIXED_SIGNAL,REVIEW_CONTENT,6
8125,content_ccf25ed65a99,client_19581e27de,10.0,0.29,LOW,0.00,keyword article,transactional,NaN,NaN,...,flat,NaN,Old,Good_Position_Low_CTR,0.817694,1,0.890617,MIXED_SIGNAL,REVIEW_CONTENT,7
24557,content_84d12054c0c0,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1635.0,11893.0,...,down,-100.0,Old,Good_Position_Low_CTR,0.815013,1,0.889008,MIXED_SIGNAL,REVIEW_CONTENT,8
1147,content_ab27c30d81f4,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1094.0,7718.0,...,stable,14.6,Old,Good_Position_Low_CTR,0.815013,1,0.889008,MIXED_SIGNAL,REVIEW_CONTENT,9
29906,content_07ce98c6085a,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1442.0,10359.0,...,down,-33.3,Old,Good_Position_Low_CTR,0.815013,1,0.889008,MIXED_SIGNAL,REVIEW_CONTENT,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Review

Each selected item is reviewed with:
- action
- reason code
- confidence
- what could make the decision wrong

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()


top20_review = top20[
[
"rank",
"content_id",
"baseline_score",
"reason_code",
"action"
]
]


top20_review["confidence_note"] = (
"Signal based decision; requires human review"
)


top20_review["what_would_make_it_wrong"] = (
"Recent improvements or missing context may change priority"
)


top20_review

/tmp/ipykernel_913/2150839786.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top20_review["confidence_note"] = (
/tmp/ipykernel_913/2150839786.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top20_review["what_would_make_it_wrong"] = (


,rank,content_id,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
26242,1,content_55a5b1c46474,1.000000,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...
24216,2,content_1b4ec72dafd4,0.998391,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...
15608,3,content_06e19c6486b0,0.937265,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...
8631,4,content_e2b702f4f92b,0.937265,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...
21984,5,content_02b0d6e30129,0.903485,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...
3723,6,content_f488400fca67,0.890617,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...
8125,7,content_ccf25ed65a99,0.890617,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...
24557,8,content_84d12054c0c0,0.889008,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...
1147,9,content_ab27c30d81f4,0.889008,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...
29906,10,content_07ce98c6085a,0.889008,MIXED_SIGNAL,REVIEW_CONTENT,Signal based decision; requires human review,Recent improvements or missing context may cha...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks and Leakage Check

Weak picks are cases where the rule may over-prioritize content.

Possible issues:
- Old content may still perform well.
- CTR can be affected by search intent.
- No future information should influence the score.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks

weak_picks = queue.tail(5)

print("Weak Picks:")
display(
weak_picks[
[
"content_id",
"baseline_score",
"reason_code",
"action"
]
]
)


# Leakage check

possible_leak_columns = [
col for col in df.columns
if "future" in col.lower()
or "label" in col.lower()
or "flag" in col.lower()
]


print("\nPossible leakage columns:")
print(possible_leak_columns)


if len(possible_leak_columns)==0:
    print("\nLeakage Check: PASS")
else:
    print("\nReview Required")

Weak Picks:


,content_id,baseline_score,reason_code,action
4029,content_023cebb114ed,0.001609,NO_ACTION,MONITOR
10346,content_e2171fbafaaf,0.001609,NO_ACTION,MONITOR
25963,content_3a8f5c52b1a0,0.001609,NO_ACTION,MONITOR
25926,content_2a843f006d86,0.001609,NO_ACTION,MONITOR
634,content_14a42157a627,0.001609,NO_ACTION,MONITOR



Possible leakage columns:
[]

Leakage Check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.